1. PREDICTIVE PARADOX: Electricity Demand Forecasting

2. OBJECTIVE: Predict Next-Hour Electricity Demand Using Historical Data, Weather Data,and Economic Data.

In [1]:
import pandas as pd
import numpy as np
import json
import warnings
warnings.filterwarnings('ignore')

from sklearn.ensemble import GradientBoostingRegressor, RandomForestRegressor
from sklearn.metrics import mean_absolute_percentage_error

Datasets:
1. PGCB date power demand
2. Weather Data
3. Economic Data

In [2]:
print("\n" + "="*65)
print("[1/9]  Loading raw datasets")
print("="*65)

pgcb_raw =pd.read_excel('PGCB_date_power_demand.xlsx')
econ_raw = pd.read_csv('economic_full_1.csv')


w_raw =pd.read_excel('weather_data.xlsx', header=2)
w_raw.columns = w_raw.iloc[0]          
w_raw =w_raw.iloc[1:].reset_index(drop=True)

print(f"PGCB: {pgcb_raw.shape}  |  {pgcb_raw['datetime'].min().date()} → {pgcb_raw['datetime'].max().date()}")
print(f"Economic:{econ_raw.shape}")
print(f"Weather:{w_raw.shape}")


[1/9]  Loading raw datasets
PGCB: (92650, 15)  |  2015-04-19 → 2025-06-17
Economic:(1516, 69)
Weather:(107304, 10)


Data Cleaning:
1. Removed duplicate timestamps 
2. Handled anomalies
3. Missing values filled 

In [3]:
print("\n[2/9] Cleaning PGCB demand data")

df =pgcb_raw.sort_values('datetime').copy()


n_before = len(df)
df = df.drop_duplicates(subset='datetime', keep='last')
print(f"  Duplicate timestamps removed : {n_before - len(df)}")


df =df.set_index('datetime').resample('1h').last()


df.loc[df['demand_mw'] < 500,   'demand_mw'] = np.nan
df.loc[df['demand_mw'] > 20000, 'demand_mw'] = np.nan


roll_q1 = df['demand_mw'].rolling(168, min_periods=24, center=True).quantile(0.05)
roll_q3 = df['demand_mw'].rolling(168, min_periods=24, center=True).quantile(0.95)
iqr = roll_q3 - roll_q1
spike_mask = (df['demand_mw'] < roll_q1 - 2.5*iqr) | (df['demand_mw'] > roll_q3 + 2.5*iqr)
df.loc[spike_mask, 'demand_mw'] = np.nan
print(f"  Spike anomalies nullified    : {spike_mask.sum()}")


df['demand_mw'] = (df['demand_mw']
 .interpolate(method='time', limit=6)
  .ffill(limit=3).bfill(limit=3))
df = df.dropna(subset=['demand_mw'])
print(f"Rows after cleaning: {len(df):,}")



[2/9] Cleaning PGCB demand data
  Duplicate timestamps removed : 432
  Spike anomalies nullified    : 13
Rows after cleaning: 88,562


Processing Weather Columns:
1. Weather Columns Joined
2. Weather NaN after Filled

In [4]:
print("\n[3/9]  Processing weather data")


WEATHER_RENAME = {
    'time': 'datetime',
    'temperature_2m (°C)' : 'temp',
    'relative_humidity_2m (%)': 'humidity',
    'apparent_temperature (°C)':'feels_like',
    'precipitation (mm)':'precip',
    'dew_point_2m (°C)': 'dew_point',
    'cloud_cover (%)': 'cloud_cover',
    'sunshine_duration (s)': 'sunshine_s',
    'wind_direction_10m (°)': 'wind_dir',
    'soil_temperature_0_to_7cm (°C)':'soil_temp',
}
w_raw = w_raw.rename(columns=WEATHER_RENAME)
w_raw['datetime'] = pd.to_datetime(w_raw['datetime'])
w_raw = w_raw.set_index('datetime').sort_index()

WEATHER_FEATS = ['temp', 'humidity', 'feels_like', 'precip',
 'dew_point', 'cloud_cover', 'sunshine_s']
for col in WEATHER_FEATS:
    if col in w_raw.columns:
        w_raw[col] = pd.to_numeric(w_raw[col], errors='coerce')

weather = w_raw[WEATHER_FEATS].resample('1h').last()


df = df.join(weather, how='left')


for col in WEATHER_FEATS:
    if col in df.columns:
        df[col] = df[col].interpolate(method='time', limit=3)

print(f"Weather columns joined  : {WEATHER_FEATS}")
print(f"Weather NaN after fill  : {df[WEATHER_FEATS].isna().sum().sum()}")


[3/9]  Processing weather data
Weather columns joined  : ['temp', 'humidity', 'feels_like', 'precip', 'dew_point', 'cloud_cover', 'sunshine_s']
Weather NaN after fill  : 0


Economic Data Joined:

In [5]:
print("\n[4/9]  Integrating economic indicators")

WANTED = [
    'GDP per capita (current US$)',
    'Urban population (% of total population)',
]
econ_sel  = econ_raw[econ_raw['Indicator Name'].isin(WANTED)].copy()
year_cols = [str(y) for y in range(2010, 2026) if str(y) in econ_raw.columns]

econ_long = econ_sel.melt(
    id_vars=['Indicator Name'], value_vars=year_cols,
    var_name='year', value_name='value'
)
econ_long['year'] = econ_long['year'].astype(int)

econ_wide = (econ_long
 .pivot_table(index='year', columns='Indicator Name', values='value')
.reset_index()
.sort_values('year')
.ffill())
econ_wide.columns = ['year', 'gdp_per_capita', 'urban_pop_pct']

df = df.reset_index()
df['year'] = df['datetime'].dt.year
df = df.merge(econ_wide, on='year', how='left').set_index('datetime')
print(f"  Joined: gdp_per_capita, urban_pop_pct  (broadcast annually → hourly)")


[4/9]  Integrating economic indicators


  Joined: gdp_per_capita, urban_pop_pct  (broadcast annually → hourly)


Engineering Features:
1. lag Features
2. Rolling Statistcs
3. Time Based Features
4. Weather And Economic Features

In [6]:
print("\n[5/9]  Engineering features")


df['hour']= df.index.hour
df['dow']= df.index.dayofweek      
df['month']= df.index.month
df['is_weekend'] = (df['dow'] >= 5).astype(int)


df['hour_sin'] = np.sin(2 * np.pi * df['hour']  / 24)
df['hour_cos'] = np.cos(2 * np.pi * df['hour']  / 24)
df['month_sin'] = np.sin(2 * np.pi * df['month'] / 12)
df['month_cos'] = np.cos(2 * np.pi * df['month'] / 12)
df['dow_sin']   = np.sin(2 * np.pi * df['dow']   / 7)
df['dow_cos']   = np.cos(2 * np.pi * df['dow']   / 7)


df['is_morning_peak'] = ((df['hour'] >= 9)  & (df['hour'] <= 12)).astype(int)
df['is_evening_peak'] = ((df['hour'] >= 18) & (df['hour'] <= 21)).astype(int)


for lag in [1, 2, 3, 6, 12, 24, 48, 168]:
    df[f'lag_{lag}'] = df['demand_mw'].shift(lag)


for w_size in [6, 24, 168]:
    rolled = df['demand_mw'].shift(1).rolling(w_size)
    df[f'roll_mean_{w_size}'] = rolled.mean()
    df[f'roll_std_{w_size}']  = rolled.std()
    df[f'roll_max_{w_size}']  = rolled.max()
    df[f'roll_min_{w_size}']  = rolled.min()


df['trend_24h'] = df['demand_mw'].shift(1) - df['demand_mw'].shift(25)
df['trend_1h']  = df['demand_mw'].shift(1) - df['demand_mw'].shift(2)



WEATHER_MODEL_COLS = ['temp', 'humidity', 'feels_like', 'precip', 'cloud_cover', 'sunshine_s']
for wf in WEATHER_MODEL_COLS:
    if wf in df.columns:
        df[f'{wf}_lag1'] = df[wf].shift(1)                          
        df[f'{wf}_roll24']  = df[wf].shift(1).rolling(24).mean()       



df['heat_stress'] = df['temp'].shift(1) * df['humidity'].shift(1) / 100
df['temp_sq'] = df['temp'].shift(1) ** 2      
df['is_rain']= (df['precip'].shift(1) > 0.5).astype(int)  
df['cool_night']  = ((df['temp'].shift(1) < 18) & (df['hour'] < 6)).astype(int)

print(f" Total columns in dataframe: {df.shape[1]}")



[5/9]  Engineering features
 Total columns in dataframe: 74


Defining The Target And Splitting:
1. Train Data
2. Test Data
3. Feature Columns

In [7]:
print("\n[6/9]  Defining target & splitting")


df['target'] = df['demand_mw'].shift(-1)
df = df.dropna(subset=['target'])


train_df = df[df.index.year < 2023].copy()
test_df  = df[df.index.year == 2023].copy()

print(f"  Train : {len(train_df):,} rows  ({train_df.index.min().date()} → {train_df.index.max().date()})")
print(f"  Test  : {len(test_df):,} rows  ({test_df.index.min().date()} → {test_df.index.max().date()})")


EXCLUDE = {
'demand_mw', 'generation_mw', 'load_shedding', 'remarks', 'year', 'target',
'gas', 'liquid_fuel', 'coal', 'hydro', 'solar', 'wind',
'india_bheramara_hvdc', 'india_tripura', 'india_adani', 'nepal',
'temp', 'humidity', 'feels_like', 'precip',
'dew_point', 'cloud_cover', 'sunshine_s', 'wind_dir', 'soil_temp',
}
FEAT = [c for c in df.columns
        if c not in EXCLUDE and pd.api.types.is_numeric_dtype(df[c])]
print(f"  Feature columns : {len(FEAT)}")


train_medians = train_df[FEAT].median()
X_tr = train_df[FEAT].fillna(train_medians)
y_tr = train_df['target']
X_te = test_df[FEAT].fillna(train_medians)
y_te = test_df['target']


[6/9]  Defining target & splitting


  Train : 66,997 rows  (2015-04-19 → 2022-12-31)
  Test  : 8,760 rows  (2023-01-01 → 2023-12-31)
  Feature columns : 52


Model Training:
      Models Used:
1. GBR:Gradient Boosting Regressor
2. Random Forest Rgressor
3. Ensemble Approach

In [ ]:
print("\n[7/9]  Training models  (this takes ~5-10 min on CPU)")


print("  Training GradientBoostingRegressor ...")
gbr = GradientBoostingRegressor(
    n_estimators =500,      
    learning_rate = 0.05,     
    max_depth = 5,        
    min_samples_leaf = 20,     
    subsample = 0.8,      
    max_features = 0.7,    
    random_state = 42,
)
gbr.fit(X_tr, y_tr)
pred_gbr = gbr.predict(X_te)
mape_gbr = mean_absolute_percentage_error(y_te, pred_gbr) * 100
print(f" GBR MAPE: {mape_gbr:.3f}%")


print("  Training RandomForestRegressor ...")
rfr = RandomForestRegressor(
    n_estimators = 300,
    max_depth  = 12,
    min_samples_leaf = 15,
    max_features = 0.6,
    random_state = 42,
    n_jobs = -1,     
)
rfr.fit(X_tr, y_tr)
pred_rfr = rfr.predict(X_te)
mape_rfr = mean_absolute_percentage_error(y_te, pred_rfr) * 100
print(f" RF  MAPE: {mape_rfr:.3f}%")


[7/9]  Training models  (this takes ~5-10 min on CPU)
  Training GradientBoostingRegressor ...


Ensemble Approach:
1. On 2023  Test DaTa

In [ ]:
print("\n[8/9]  Ensemble evaluation on 2023 hold-out")


pred_ens = 0.60 * pred_gbr + 0.40 * pred_rfr
mape_ens = mean_absolute_percentage_error(y_te, pred_ens) * 100
mae      = np.mean(np.abs(y_te.values - pred_ens))
rmse     = np.sqrt(np.mean((y_te.values - pred_ens) ** 2))


test_results = test_df[['demand_mw', 'target']].copy()
test_results['pred'] = pred_ens
test_results['ape'] = np.abs(test_results['target'] - test_results['pred']) / test_results['target'] * 100
monthly_mape = test_results.groupby(test_results.index.month)['ape'].mean().round(2)


print(f"GradientBoosting MAPE:{mape_gbr:7.3f}%")
print(f"RandomForest MAPE:{mape_rfr:7.3f}%")
print(f"ENSEMBLE MAPE (60/40): {mape_ens:7.3f}%")
print(f"Ensemble MAE:{mae:7.1f} MW")
print(f"Ensemble RMSE:{rmse:7.1f} MW")


print("\n  Monthly MAPE breakdown (2023):")
month_names = ['Jan','Feb','Mar','Apr','May','Jun','Jul','Aug','Sep','Oct','Nov','Dec']
for m, mape_m in monthly_mape.items():
    bar = '█' * int(mape_m / 0.5)
    print(f"{month_names[m-1]:3s}  {mape_m:5.2f}%  {bar}")

if mape_ens < 3:
    interp = "Excellent — industrial-grade grid forecasting accuracy."
elif mape_ens < 5:
    interp = "Very good — suitable for real-time grid operational decisions."
elif mape_ens < 8:
    interp = "Good — suitable for day-ahead planning and scheduling."
else:
    interp = "Moderate — consider adding holiday calendar or more weather stations."
print(f"\n  Interpretation: {interp}")


[8/9]  Ensemble evaluation on 2023 hold-out
GradientBoosting MAPE:  3.899%
RandomForest MAPE:  4.238%
ENSEMBLE MAPE (60/40):   3.950%
Ensemble MAE:  339.4 MW
Ensemble RMSE:  617.5 MW

  Monthly MAPE breakdown (2023):
Jan   2.23%  ████
Feb   2.41%  ████
Mar   5.81%  ███████████
Apr   5.22%  ██████████
May   8.59%  █████████████████
Jun   3.50%  ███████
Jul   5.43%  ██████████
Aug   3.62%  ███████
Sep   2.51%  █████
Oct   2.43%  ████
Nov   2.85%  █████
Dec   2.59%  █████

  Interpretation: Very good — suitable for real-time grid operational decisions.


Final Result:
The Final Test MAPE (2023 hold-out) is 3.95%

In [ ]:
print("\n[9/9]  Feature importance & saving outputs")

fi = pd.Series(gbr.feature_importances_, index=FEAT)
print("\n  Top 20 Feature Importances (GBR):")
top20 = fi.nlargest(20)
for feat, imp in top20.items():
    bar = '█' * int(imp * 200)
    print(f"    {feat:<30s}  {imp:.4f}  {bar}")


fi_df = (pd.DataFrame({'feature': FEAT, 'importance': gbr.feature_importances_})
         .sort_values('importance', ascending=False))
fi_df.to_csv('feature_importances.csv', index=False)

test_results.to_csv('test_predictions_2023.csv')

summary = {
    'model':'GBR + RandomForest Ensemble (60/40)',
    'version':'v2_with_weather',
    'train_period':f"{train_df.index.min().date()} to {train_df.index.max().date()}",
    'test_period':f"{test_df.index.min().date()} to {test_df.index.max().date()}",
    'n_features':len(FEAT),
    'feature_list':FEAT,
    'n_train':int(len(X_tr)),
    'n_test':int(len(X_te)),
    'MAPE_GBR_%':round(mape_gbr, 4),
    'MAPE_RF_%':round(mape_rfr, 4),
    'MAPE_Ensemble_%':round(mape_ens, 4),
    'MAE_MW':round(float(mae), 2),
    'RMSE_MW':round(float(rmse), 2),
    'interpretation':interp,
    'top5_features':fi.nlargest(5).index.tolist(),
    'monthly_mape':{month_names[m-1]: v for m, v in monthly_mape.items()},
}
with open('model_summary.json', 'w') as f:
    json.dump(summary, f, indent=2)

print("\n  Files saved:")
print("feature_importances.csv")
print("test_predictions_2023.csv")
print("model_summary.json")

print("\n" + "="*65)
print(f"FINAL TEST MAPE (2023 hold-out) : {mape_ens:.3f}%")
print("="*65 + "\n")



[9/9]  Feature importance & saving outputs

  Top 20 Feature Importances (GBR):
    lag_1                           0.6370  ███████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████
    lag_24                          0.1339  ██████████████████████████
    roll_mean_24                    0.0953  ███████████████████
    hour_sin                        0.0256  █████
    hour_cos                        0.0218  ████
    lag_48                          0.0186  ███
    hour                            0.0090  █
    trend_24h                       0.0086  █
    lag_2                           0.0082  █
    roll_max_24                     0.0075  █
    trend_1h                        0.0043  
    lag_168                         0.0036  
    roll_min_6                      0.0026  
    sunshine_s_lag1                 0.0026  
    lag_12                          0.0023  
    roll_max_168                    0.0023  
    lag_3  